<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
#%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

In [3]:
# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged','7-wonders']
IT = 10
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
OVERWRITE = ['']
MAX_RETRY = 10
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/estimation/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = 'estimation/'
    N_GPU_LAYERS = 20

In [4]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os



# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q6_K.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"bartowski/Qwen2.5-7B-Instruct-GGUF",
             'filename': "Qwen2.5-7B-Instruct-Q6_K.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'gemma': {'repo_id':"bartowski/google_gemma-3n-E4B-it-GGUF",
                'filename':"google_gemma-3n-E4B-it-Q6_K.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': None
             },
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=N_GPU_LAYERS, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=VERBOSE,
                            force_download=True,
                            enable_thinking=True)


def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]



prompts = {}
rfs={}

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce GTX 1070, compute capability 6.1, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce GTX 1070) - 7685 MiB free
llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /home/prometheus/.cache/huggingface/hub/models--bartowski--Meta-Llama-3.1-8B-Instruct-GGUF/snapshots/bf5b95e96dac0462e2a09145ec66cae9a3f12067/./Meta-Llama-3.1-8B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_mo

In [5]:
import re
test1 = 'mechanics: [A, B, ..., Z]'
test2 = 'a\nbhst\nOverall Complexity: 4.2'
test3 ='tsr\n*optimal number of pLayers: 10*'
test4 ='tsr\noptimal Player count: 1-2'
test_dur_1 = 'sthser\nduration: 10-20'
test_dur_2 = 'sthser\nduration: 120-30 minutes'

regex = {'mechanics': r"(?i)^.*\bmechanics\s*:\s*\[.*\]\s*$",
         'complexity': r'(?i)^.*\bcomplexity:\s*[1-5]\.\d\s*$',
         'player': r'(?i)^.*\bplayer.*:\s*\d+',
         'duration': r'(?i)^.*\bduration.*:\s*(\d+)(?:-(\d+))?'

}
def check_output(prompt, output):
    if(prompt == 'all'):
        try:
            out_dic = json.loads(output)
            if('answer' in out_dic):
                return True
            else:
                return False
        except:
            return False
    else:
        last_line = output.split('\n')[-1]
        return bool(re.match(regex[prompt], last_line))


print(check_output('mechanics',test1))
print(check_output('complexity',test2))
print(check_output('player',test3))
print(check_output('player',test4))
print(check_output('duration',test_dur_1))
print(check_output('duration',test_dur_2))


True
True
True
True
True
True


# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [6]:
prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1.0‑5.0).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

<<<JSON schema>>>
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}


<<<EXAMPLE>>>
{
  "reasoning": "The rules describe moving pieces on a grid, controlling regions, and a simple scoring system. These map to Area Control and Hand Management. The rule density is low and decisions are straightforward, so complexity is around 1.8. With only 30 tokens and a small board, the game works best with 2‑3 players; 2 is optimal for maximum interaction. Each turn shifts control of a few squares, and a full game finishes in roughly 30 minutes.",
  "answer": {
    "mechanics": ["Area Control", "Hand Management"],
    "complexity": 1.8,
    "optimal player count": 2,
    "duration": 30
  }
}
"""

rfs['all'] = {"type": "json_object",
          "schema": {
              "type": "object",
              "properties": {
                  "reasoning": {"type": "string"},
                  "answer": {"type": "object", "properties": {
                      "mechanics": {"type": "array", "items": {"type": "string"}},
                      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
                      "optimal player count": {"type": "number"},
                      "duration": {"type": "number"},
                      "required": ["mechanics", "complexity", "optimal player count", "duration"]
                      }
                    }
                  },
              "required": ["reasoning", "answer"]
              }
          }



## Estimating each parameter separately

### Mechanics

In [7]:
prompts['mechanics'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s). Use only mechanics present in the list below.

here is a complete list of the available BGG mechanics:
[Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction,
"Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution,
Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events,
Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management,
Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato,
"I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction,
Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering,
Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control]

after your reasoning output one last line with the chosen mechanics.

**Final Output**
mechanics: [A, B, ..., Z]
"""
rfs['mechanics'] = {"type": "json_object",
                   "schema": {
                       "type": "object",
                       "properties": {
                           "reasoning": {"type": "string"},
                           "answer": {"type": "array", "items": {"type": "string"}}
                       },
                       "required": ["reasoning", "answer"]
                   }
                  }


### Complexity rating

In [8]:
prompts['complexity'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

After your reasoning output one last line with the overall complexity rating.

**Final Output**
Overall Complexity: W.W"""

rfs['mechanics'] = {"type": "json_object",
                   "schema": {
                       "type": "object",
                       "properties": {
                           "reasoning": {"type": "string"},
                           "answer": {"type": "array", "items": {"type": "string"}}
                       },
                       "required": ["reasoning", "answer"]
                   }
                  }

### Optimal player count

In [9]:
prompts['player'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count (a single number not a range) based on your observations

After your reasoning output one last line with the optimal player count.

**Final Output**
Optimal player count: X
"""

### Game duration

In [10]:
prompts['duration'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

After your reasoning output one final line wit the expected game duration.

**Final Output**
Average game duration: X minutes
"""

## Execution

In [11]:
# select only prompt not executed
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                               or f'{CHOSEN}_{g}.json' in OVERWRITE]
if to_do == []:
    print('Nothing to do')

for g in to_do:
    print(g)
    output_dict = {p: {it: '' for it in range(IT)} for p in prompts.keys()}
    for p,it in tqdm([(p,it) for p in reversed(list(prompts.keys())) for it in range(IT)]):
        for retry in range(MAX_RETRY):
            if(retry > 0 and VERBOSE):
                print(f'Retry {retry}')
            rulebook = requests.get(BASE_URL +g+'.txt').text
            name = g.replace('_',' ')
            out = model.create_chat_completion(generate_message(prompts[p], f'Here is the full rulebook of the game {name}:\n'+rulebook),
                                               temperature=models[CHOSEN]['temperature'],
                                               response_format = rfs['all'] if p == 'all' else None,
                                               )['choices'][0]['message']['content']
            if(check_output(p,out)):
                break
        output_dict[p][it] = out
        if(VERBOSE):
            print(p,'-',it)
            print(out)

    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

7-wonders


  0%|                                                    | 0/50 [00:00<?, ?it/s]llama_perf_context_print:        load time =     684.38 ms
llama_perf_context_print: prompt eval time =     683.41 ms /   138 tokens (    4.95 ms per token,   201.93 tokens per second)
llama_perf_context_print:        eval time =   55040.99 ms /   352 runs   (  156.37 ms per token,     6.40 tokens per second)
llama_perf_context_print:       total time =   56141.13 ms /   490 tokens
llama_perf_context_print:    graphs reused =        350
  2%|▉                                           | 1/50 [00:56<46:02, 56.39s/it]Llama.generate: 137 prefix-match hit, remaining 1 prompt tokens to eval


duration - 0
Unfortunately, it seems that the rulebook you provided is not accessible. However, I can provide a general analysis of the game 7 Wonders based on its mechanics and gameplay.

**Assessment of a typical game turn:**

In 7 Wonders, a typical game turn involves players taking individual turns to draw cards, play cards, and pass on resources. Players have a hand of cards that represent different structures, technologies, and military units. Each turn, a player draws a card, then plays one card from their hand, and finally passes on a certain number of resources (stones, glass, papyrus, and manna) to the next player.

**Evaluation of complexity:**

The complexity of actions in 7 Wonders is moderate. Players need to manage their hand, keep track of available resources, and make tactical decisions about which cards to play and when to pass on resources. However, the game is relatively straightforward, and most players can learn the mechanics quickly.

**Evaluation of progress tow

llama_perf_context_print:        load time =     684.38 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   48779.67 ms /   314 runs   (  155.35 ms per token,     6.44 tokens per second)
llama_perf_context_print:       total time =   49143.65 ms /   315 tokens
llama_perf_context_print:    graphs reused =        312
Llama.generate: 137 prefix-match hit, remaining 1 prompt tokens to eval


Retry 1


...

Retry 1


llama_perf_context_print:        load time =     684.38 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    5773.18 ms /    35 runs   (  164.95 ms per token,     6.06 tokens per second)
llama_perf_context_print:       total time =    6564.40 ms /    36 tokens
llama_perf_context_print:    graphs reused =         35
Llama.generate: 526 prefix-match hit, remaining 1 prompt tokens to eval


Retry 2


 82%|████████████████████████████████▊       | 41/50 [1:11:09<15:37, 104.14s/it]


KeyboardInterrupt: 